# Klasifikasi DemogPairs Menggunakan ViT (Wajah dan Umur) & Gaussian Naive Bayes

In [1]:
import numpy as np
import utils as u
import joblib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from tqdm import tqdm

joblib.parallel_backend('threading')

## Load Dataset

In [3]:
data = u.load_demogpairs()
pd.DataFrame(data)

## Load Fitur

In [5]:
face_features = joblib.load('features/demogpairs_vit-face.pkl')
age_features = joblib.load('features/demogpairs_vit-age.pkl')
features = {}
for d in tqdm(data):
    key = d['image_path']
    features[key] = np.array(list(face_features[key]) + list(age_features[key]))
print('Jumlah fitur per gambar:', np.array(features[list(features.keys())[0]]).shape[0])

Jumlah fitur per gambar: 1536


## Split Data

In [7]:
X = np.array([features[d['image_path']] for d in data])
y = np.array([d['label_idx'] for d in data])
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
print((len(X_train), len(X_test)))

(8640, 2160)


## Kombinasi Parameter

In [9]:
var_smoothing_values = np.logspace(-9, 2, 40)  # dari 1e-9 sampai 1e2, 40 nilai

grid_params = [
    {
        'scaler': [None, MinMaxScaler()],
        'pca': [None, PCA(n_components=0.5), PCA(n_components=0.75)],
        
        'classifier': [GaussianNB()],
        'classifier__var_smoothing': var_smoothing_values
    },
]

pipeline = Pipeline(steps=[
    ('scaler', None),
    ('pca', None),
    ('classifier', None)
])

skv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy', 
    'f1': 'f1_macro', 
    'precision': 'precision_macro', 
    'recall': 'recall_macro',
    'roc_auc_ovr': 'roc_auc_ovr'
}

grid_models = {}
for params in grid_params:
    key = str(params['classifier'][0]).split('(')[0]
    grid_models[key] = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        cv=skv, refit='accuracy',
        scoring=scoring, n_jobs=int(joblib.cpu_count() * 0.6),
        verbose=1, error_score='raise',
        return_train_score=True
    )
    print(f'{key}: {len(ParameterGrid(params))} kombinasi')

GaussianNB: 240 kombinasi


## Klasifikasi

In [11]:
evaluation_results, fold_results = u.evaluate_models(
    grid_models,
    X_train, y_train,
    X_test, y_test,
    model_prefix='models/clf_demogpairs_gnb_vit-face-age_',
    results_path='results/demogpairs_gnb_vit-face-age_'
)

sorted_results = pd.DataFrame(evaluation_results).sort_values(by='test_accuracy', ascending=False).to_dict('records')
u.html_br()
_dtable = u.display_table(sorted_results)

Evaluating: GaussianNB

##### Best Parameters
{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.011253355826007646), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}

##### Test Results
Accuracy  : 0.8314814814814815
Precision : 0.8343036484691989
Recall    : 0.8314814814814815
F1 Score  : 0.8316522462782968
              precision    recall  f1-score   support

           0       0.82      0.90      0.86       360
           1       0.91      0.86      0.88       360
           2       0.79      0.78      0.78       360
           3       0.90      0.87      0.88       360
           4       0.84      0.75      0.79       360
           5       0.76      0.84      0.79       360

    accuracy                           0.83      2160
   macro avg       0.83      0.83      0.83      2160
weighted avg       0.83      0.83      0.83      2160


Class
    Accuracy
    Precision
    Recall
    F1-Score
  
  
    Black_Males
    0.9495370370370371
    0.8177215189873418
    0.897

In [12]:
model, training_time = u.load_object('models/clf_demogpairs_gnb_vit-face-age_GaussianNB.pkl')
u.h(5, 'Waktu Pelatihan (Jobs)')
u.seconds_to_time(round(training_time))


##### Waktu Pelatihan (Jobs)


In [13]:
u.h(5, 'Waktu Pelatihan')
times = [fr['Train Time Mean'] * 5 for fr in fold_results]
u.seconds_to_time(round(np.sum(times) + model.refit_time_))


##### Waktu Pelatihan


In [14]:
_dtable = u.display_table(fold_results, n_items=[4, 4], column_widths=['5%', '45%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%'])

No
    Params
    Fold 1
    Fold 2
    Fold 3
    Fold 4
    Fold 5
    Accuracy Mean
    F1 Score Mean
    Precision Mean
    Recall Mean
    Train Time Mean
  
  
    1
    {'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.011253355826007646), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}
    0.8519
    0.8328
    0.8414
    0.8432
    0.8478
    0.8434
    0.8432
    0.8466
    0.8434
    9.2913
  
  
    2
    {'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.0058780160722749115), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}
    0.8501
    0.8351
    0.842
    0.8449
    0.8438
    0.8432
    0.8428
    0.8445
    0.8432
    10.4237
  
  
    3
    {'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.0058780160722749115), 'pca': 'PCA', 'scaler': None}
    0.8478
    0.8328
    0.8409
    0.8426
    0.8478
    0.8424
    0.842
    0.8442
    0.8424
    11.5458
  
  
    4
    {'classifier': 'GaussianNB', 'classifier__var_smoothing': 